# 04 - Modelling

## Data sources
- **ML-HydPARK** (Zenodo, v0.0.5): cleaned experimental thermodynamic data for metal hydrides (~770 entries)
- **ElementalH_Ef**: experimental elemental hydride formation energies, used for compositional feature engineering
- **ElementalH_Ef_MP**: Materials Project DFT calculations - not used (more missing values, DFT approximations)

## Target Variables
- `Temperature_oC`
- `Hydrogen_Weight_Percent`

## Goals
- Investigate baseline models for each dataset
- Evaluation and fine tuning of chosen models
- SHAP analysis for model interpretability

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
from xgboost import XGBRegressor
import shap
import chemparse
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV

In [2]:
# Loading dataset A
X_A_train = pd.read_csv('../data/processed/splits/X_A_train.csv')
X_A_val = pd.read_csv('../data/processed/splits/X_A_val.csv')
X_A_test = pd.read_csv('../data/processed/splits/X_A_test.csv')
y_A_train = pd.read_csv('../data/processed/splits/y_A_train.csv').squeeze()
y_A_val = pd.read_csv('../data/processed/splits/y_A_val.csv').squeeze()
y_A_test = pd.read_csv('../data/processed/splits/y_A_test.csv').squeeze()

# Loading dataset B
X_B_train = pd.read_csv('../data/processed/splits/X_B_train.csv')
X_B_val = pd.read_csv('../data/processed/splits/X_B_val.csv')
X_B_test = pd.read_csv('../data/processed/splits/X_B_test.csv')
y_B_train = pd.read_csv('../data/processed/splits/y_B_train.csv').squeeze()
y_B_val = pd.read_csv('../data/processed/splits/y_B_val.csv').squeeze()
y_B_test = pd.read_csv('../data/processed/splits/y_B_test.csv').squeeze()

print(X_A_train.shape, X_A_val.shape, X_A_test.shape)
print(X_B_train.shape, X_B_val.shape, X_B_test.shape)


(518, 11) (111, 11) (112, 11)
(266, 10) (57, 10) (58, 10)


## Dataset A

Target variable: `Hydrogen_Weight_Percent`

### Linear Regression: baseline model

In [3]:
# Model instantiation, training and validation
lr_A = LinearRegression()
lr_A.fit(X_A_train, y_A_train)
y_A_val_pred_lr = lr_A.predict(X_A_val)


# Evaluation
print('R²:', r2_score(y_A_val, y_A_val_pred_lr))
print('RMSE:', np.sqrt(mean_squared_error(y_A_val, y_A_val_pred_lr)))
print('MAE:', mean_absolute_error(y_A_val, y_A_val_pred_lr))

R²: 0.29595540128489306
RMSE: 0.5387057686587513
MAE: 0.3392752642821463


### Random Forest: baseline model

In [4]:
# Model instantiation, training and validation
rf_A = RandomForestRegressor(random_state=42)
rf_A.fit(X_A_train, y_A_train)
y_A_val_pred_rf = rf_A.predict(X_A_val)

# Evaluation
print('R²:', r2_score(y_A_val, y_A_val_pred_rf))
print('RMSE:', np.sqrt(mean_squared_error(y_A_val, y_A_val_pred_rf)))
print('MAE:', mean_absolute_error(y_A_val, y_A_val_pred_rf))

R²: 0.5011263811654405
RMSE: 0.45346817161085257
MAE: 0.30237191642493616


### XGBoost Regressor: baseline model

In [5]:
# Model instantiation, training and validation
xgb_A = XGBRegressor()
xgb_A.fit(X_A_train, y_A_train)
y_A_val_pred_xgb = xgb_A.predict(X_A_val)


# Evaluation
print('R²:', r2_score(y_A_val, y_A_val_pred_xgb))
print('RMSE:', np.sqrt(mean_squared_error(y_A_val, y_A_val_pred_xgb)))
print('MAE:', mean_absolute_error(y_A_val, y_A_val_pred_xgb))

R²: 0.4207742780016551
RMSE: 0.48862474909310166
MAE: 0.3333662903103805


## Baseline Models Comparison

Random Forest is the best performing model compared to XGBoost and Linear Regression:

| Model | R² | RMSE | MAE |
|---|---|---|---|
| Linear Regression | 0.296 | 0.539 | 0.339 |
| Random Forest | 0.501 | 0.453 | 0.302 |
| XGBoost | 0.421 | 0.489 | 0.333 |

Random Forest is the best model with an R²=0.501, which means that the vanilla model explains approximately half of the variance in `Hydrogen_Weight_Percent`.

## Next Steps

Feature engineering: adding the `ElementalH_Ef` to the features could help the model better understand and distinguish between hydrogen storage materials with similar enthalpy of formation but different elemental composition.

### Feature Engineering

Exploring the `ElementalH_Ef` and `ElementalH_Ef_MP` datasets to understand how to leverage them as features.

In [6]:
# Loading the ElementalH_Ef dataset
df_H_Ef = pd.read_csv('../data/raw/ElementalH_Ef.csv')
df_H_Ef.head()

,No,Species,Ef
0,1.0,NaN,NaN
1,2.0,NaN,NaN
2,3.0,Li,-0.41
3,4.0,Be,-0.04
4,5.0,B,-0.08


In [7]:
# Checking shape and missing values in df_H_Ef
print(df_H_Ef.shape)
print(df_H_Ef.isna().sum())

(94, 3)
No         1
Species    8
Ef         8
dtype: int64


In [8]:
# Loading the ElementalH_Ef_MP dataset
df_H_Ef_MP = pd.read_csv('../data/raw/ElementalH_Ef_MP.csv')
df_H_Ef_MP.head()

,Unnamed: 0,No,Species,Ef
0,0,1.0,NaN,NaN
1,1,2.0,NaN,NaN
2,2,3.0,Li,-0.482520
3,3,4.0,Be,-0.149158
4,4,5.0,B,-0.174701


In [9]:
# Checking shape and missing values in df_H_Ef_MP
print(df_H_Ef_MP.shape)
print(df_H_Ef_MP.isna().sum())

(94, 4)
Unnamed: 0     0
No             1
Species        8
Ef            15
dtype: int64


`ElementalH_Ef` contains experimental formation energies, whereas `ElementalH_Ef_MP` comes from Materials Project DFT calculations.

`ElementalH_Ef` is preferred for two reasons: it contains fewer missing values (8 vs 15), and it is methodologically more consistent with ML-HydPARK, which is also experimentally derived. Using experimental features alongside experimental target data reduces the risk of systematic bias introduced by DFT approximations.

In [10]:
# Dropping missing values from df_H_Ef
df_H_Ef = df_H_Ef.dropna()

# Creating a dictionary that has elements as keys and their H as value
ef_dict = dict(zip(df_H_Ef['Species'], df_H_Ef['Ef']))
ef_dict


{'Li': -0.41,
 'Be': -0.04,
 'B': -0.08,
 'C': -0.14,
 'N': -0.37,
 'O': -1.18,
 'F': -1.62,
 'Na': -0.25,
 'Mg': -0.18,
 'Al': -0.01,
 'Si': 0.03,
 'P': 0.63,
 'S': -0.24,
 'Cl': -0.67,
 'Ar': 0.0,
 'K': -0.2,
 'Ca': -0.57,
 'Sc': -0.67,
 'Ti': -0.49,
 'V': -0.2,
 'Cr': 0.02,
 'Mn': 0.24,
 'Fe': 0.0,
 'Co': 0.16,
 'Ni': -0.05,
 'Cu': 0.13,
 'Zn': 0.41,
 'Ga': 0.36,
 'Ge': 0.12,
 'As': 0.68,
 'Se': 0.79,
 'Br': -0.39,
 'Kr': 0.0,
 'Rb': -0.16,
 'Sr': -0.54,
 'Y': -0.7,
 'Zr': -0.56,
 'Nb': -0.23,
 'Mo': 0.4,
 'Tc': 0.47,
 'Ru': 0.45,
 'Rh': 0.04,
 'Pd': -0.07,
 'Ag': 0.58,
 'Cd': 0.46,
 'In': 0.4,
 'Sn': 0.19,
 'Sb': 0.59,
 'Te': 0.74,
 'I': -0.07,
 'Xe': 0.0,
 'Cs': -0.16,
 'Ba': -0.49,
 'La': -0.63,
 'Ce': -0.59,
 'Pr': -0.61,
 'Nd': -0.63,
 'Pm': -0.52,
 'Sm': -0.65,
 'Eu': -0.64,
 'Gd': -0.68,
 'Tb': -0.47,
 'Dy': -0.69,
 'Ho': -0.69,
 'Er': -0.69,
 'Tm': -0.69,
 'Yb': 0.62,
 'Lu': -0.68,
 'Hf': -0.49,
 'Ta': -0.15,
 'W': 0.62,
 'Re': 0.73,
 'Os': 0.73,
 'Ir': 0.52,
 'Pt': 0.54,
 '

The `Ef` values in `ef_dict` are consistent with known hydride chemistry: elements such as La, Ce, Zr and Ti have negative Ef (stable hydride formers), while Mn, Fe and Cr are near zero or positive (unstable hydrides). This confirms the dictionary can be used as a meaningful feature for the model.

In [11]:
# Creating the function to calculate Ef_weighted
def compute_ef_weighted(formula, ef_dict):
    elements = chemparse.parse_formula(formula)
    total = sum(elements.values())
    weighted_ef = 0
    for element, count in elements.items():
        if element in ef_dict:
            weighted_ef += (count / total) * ef_dict[element]
    return weighted_ef

In [12]:
# Testing the function on LaNi5
compute_ef_weighted('LaNi5', ef_dict)

-0.14666666666666667

### Verifying the output of compute_ef_weighted

For `LaNi5` (total atoms = 6):

- Ef(La) = -0.63 → contribution: (1/6) × (-0.63) = -0.105
- Ef(Ni) = -0.05 → contribution: (5/6) × (-0.05) = -0.042

Ef_weighted = -0.105 + (-0.042) = -0.147 ✓

In [13]:
# Reload original cleaned data to access Composition_Formula
df_raw = pd.read_csv('../data/processed/ML-HYDPARK_eda_cleaned.csv')

# Recreate Dataset A filtering. Same as notebook 03
df_A_orig = df_raw.dropna(subset=['Hydrogen_Weight_Percent']).copy().reset_index(drop=True)

In [14]:
# Compute weighted average Ef for each compound
ef_values = []
for formula in df_A_orig['Composition_Formula']:
    ef = compute_ef_weighted(formula, ef_dict)
    ef_values.append(ef)

df_A_orig['Ef_weighted'] = ef_values

df_A_orig[['Composition_Formula', 'Ef_weighted']].head()


,Composition_Formula,Ef_weighted
0,Th2Al,-0.323333
1,Ti2Cu,-0.283333
2,Zr2Cu,-0.330000
3,Zr2Ni,-0.390000
4,Mg2Ni,-0.136667


In [15]:
# Recreate the same train/val/test split indices as notebook 03
# Assumption: split indices must match those used in notebook 03 (random_state=42, test_size=0.30/0.50)
idx = df_A_orig.index
idx_train, idx_temp = train_test_split(idx, test_size=0.30, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42)

# Add Ef_weighted to each split
X_A_train['Ef_weighted'] = df_A_orig.loc[idx_train, 'Ef_weighted'].values
X_A_val['Ef_weighted'] = df_A_orig.loc[idx_val, 'Ef_weighted'].values
X_A_test['Ef_weighted'] = df_A_orig.loc[idx_test, 'Ef_weighted'].values

print(X_A_train.shape)

(518, 12)


In [16]:
# Checking the shape of train/val/test splits
print(X_A_train.shape, X_A_val.shape, X_A_test.shape)

(518, 12) (111, 12) (112, 12)


### Baseline models with added Ef_weighted feature

### Linear Regression

In [17]:
# Model instantiation, training and validation
lr_A = LinearRegression()
lr_A.fit(X_A_train, y_A_train)
y_A_val_pred_lr = lr_A.predict(X_A_val)


# Evaluation
print('R²:', r2_score(y_A_val, y_A_val_pred_lr))
print('RMSE:', np.sqrt(mean_squared_error(y_A_val, y_A_val_pred_lr)))
print('MAE:', mean_absolute_error(y_A_val, y_A_val_pred_lr))

R²: 0.412508838360367
RMSE: 0.4920986904658205
MAE: 0.3216677511396114


### Random Forest

In [18]:
# Model instantiation, training and validation
rf_A = RandomForestRegressor(random_state=42)
rf_A.fit(X_A_train, y_A_train)
y_A_val_pred_rf = rf_A.predict(X_A_val)

# Evaluation
print('R²:', r2_score(y_A_val, y_A_val_pred_rf))
print('RMSE:', np.sqrt(mean_squared_error(y_A_val, y_A_val_pred_rf)))
print('MAE:', mean_absolute_error(y_A_val, y_A_val_pred_rf))

R²: 0.6179720529439812
RMSE: 0.3968250758863964
MAE: 0.26162568563340377


### XGBoost

In [19]:
# Model instantiation, training and validation
xgb_A = XGBRegressor()
xgb_A.fit(X_A_train, y_A_train)
y_A_val_pred_xgb = xgb_A.predict(X_A_val)


# Evaluation
print('R²:', r2_score(y_A_val, y_A_val_pred_xgb))
print('RMSE:', np.sqrt(mean_squared_error(y_A_val, y_A_val_pred_xgb)))
print('MAE:', mean_absolute_error(y_A_val, y_A_val_pred_xgb))

R²: 0.5966240849659128
RMSE: 0.4077617856609055
MAE: 0.2809318794042477


### Comparison between models before and after feature engineering

All three models show a noticeably higher R² value, with Random Forest still being the best model among the three.

| Model | R² (baseline) | R² (+ Ef_weighted) |
|---|---|---|
| Linear Regression | 0.296 | 0.413 |
| Random Forest | 0.501 | **0.618** |
| XGBoost | 0.421 | 0.597 |

**Next step**: fine tuning of Random Forest model.

### Random Forest Fine Tuning

In [20]:
# Hyperparameter grid for RandomizedSearchCV
# max_depth capped at 30 to prevent overfitting
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_random = RandomizedSearchCV(
    estimator=RandomForestRegressor(),
    param_distributions=param_grid,
    n_iter=100,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

rf_random.fit(X_A_train, y_A_train)

print('Best parameters:', rf_random.best_params_)
print('Best R²:', rf_random.best_score_)


Best parameters: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 30}
Best R²: 0.5941811492610329


In [21]:
rf_tuned = rf_random.best_estimator_
y_A_val_pred_rf_tuned = rf_tuned.predict(X_A_val)

print('R²:', r2_score(y_A_val, y_A_val_pred_rf_tuned))
print('RMSE:', np.sqrt(mean_squared_error(y_A_val, y_A_val_pred_rf_tuned)))
print('MAE:', mean_absolute_error(y_A_val, y_A_val_pred_rf_tuned))


R²: 0.5298356096776553
RMSE: 0.44022672835337046
MAE: 0.28584674639053365


### Dataset A - Random Forest Optimisation

Hyperparameter tuning via `RandomizedSearchCV` (n_iter=100, cv=5) did not improve over the vanilla model: the best tuned R²=0.530 vs vanilla R²=0.618 on the validation set.

With only 518 training samples, hyperparameter tuning via cross-validation did not generalise well to the validation set, likely due to high variance with small data. The vanilla Random Forest is retained as the final model for Dataset A.